# 01 資料前處理

處理凱米颱風（2024-07-25）與梅雨（2024-05-28）的降雨觀測資料，進行篩選、清洗與空間轉換。

In [15]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import os

## 通用資料處理函數

In [16]:
def process_rainfall_data(
    csv_path,
    target_time,
    output_path,
    target_counties=["花蓮縣", "宜蘭縣"]
):
    """
    通用降雨資料處理函數
    
    Parameters:
    -----------
    csv_path : str
        原始 CSV 檔案路徑
    target_time : str
        目標時間字串，格式為 "YYYY/MM/DD HH:MM:SS AM"
    output_path : str
        輸出 GeoJSON 檔案路徑
    target_counties : list
        要保留的縣市列表，預設為 ["花蓮縣", "宜蘭縣"]
    """
    print(f"\n{'='*60}")
    print(f"處理檔案: {csv_path}")
    print(f"目標時間: {target_time}")
    print(f"{'='*60}")
    
    # 1. 讀取 CSV
    print("\n[Step 1] 讀取 CSV 檔案...")
    df = pd.read_csv(csv_path)
    print(f"  原始資料筆數: {len(df)}")
    
    # 2. 篩選縣市
    print(f"\n[Step 2] 篩選縣市: {target_counties}...")
    df_filtered = df[df['CountyName'].isin(target_counties)].copy()
    print(f"  篩選後資料筆數: {len(df_filtered)}")
    
    # 3. 鎖定時間
    print(f"\n[Step 3] 篩選時間: {target_time}...")
    # 確保 DateTime 欄位為字串格式以便比對
    df_filtered['DateTime'] = df_filtered['DateTime'].astype(str)
    df_time_filtered = df_filtered[df_filtered['DateTime'] == target_time].copy()
    print(f"  時間篩選後資料筆數: {len(df_time_filtered)}")
    
    if len(df_time_filtered) == 0:
        print("  ⚠️ 警告: 該時間點無資料，請檢查時間格式是否正確！")
        return None
    
    # 4. 清洗數值（過濾異常值與無雨站點）
    print("\n[Step 4] 清洗數值（過濾 Past1hr 為 -998 或 <= 0）...")
    before_clean = len(df_time_filtered)
    df_clean = df_time_filtered[
        (df_time_filtered['Past1hr'] != -998) & 
        (df_time_filtered['Past1hr'] > 0)
    ].copy()
    after_clean = len(df_clean)
    print(f"  清洗前資料筆數: {before_clean}")
    print(f"  清洗後資料筆數: {after_clean}")
    print(f"  移除異常/無雨站點數: {before_clean - after_clean}")
    
    if len(df_clean) == 0:
        print("  ⚠️ 警告: 清洗後無有效資料！")
        return None
    
    # 5. 空間轉換
    print("\n[Step 5] 建立 GeoDataFrame 並進行座標轉換...")
    
    # 使用經緯度建立 geometry
    geometry = [Point(xy) for xy in zip(df_clean['StationLongitude'], df_clean['StationLatitude'])]
    gdf = gpd.GeoDataFrame(df_clean, geometry=geometry, crs="EPSG:4326")
    print(f"  原始 CRS: EPSG:4326 (WGS84)")
    
    # 轉換至台灣二度分帶 EPSG:3826
    gdf_3826 = gdf.to_crs("EPSG:3826")
    print(f"  轉換後 CRS: EPSG:3826 (TWD97 二度分帶)")
    
    # 新增 easting 與 northing 欄位
    gdf_3826['easting'] = gdf_3826.geometry.x
    gdf_3826['northing'] = gdf_3826.geometry.y
    print(f"  已新增 easting 與 northing 欄位")
    
    # 顯示資料統計
    print(f"\n[資料統計]")
    print(f"  Past1hr 平均值: {gdf_3826['Past1hr'].mean():.2f} mm")
    print(f"  Past1hr 最大值: {gdf_3826['Past1hr'].max():.2f} mm")
    print(f"  Past1hr 最小值: {gdf_3826['Past1hr'].min():.2f} mm")
    
    # 6. 匯出檔案
    print(f"\n[Step 6] 匯出至 {output_path}...")
    
    # 確保輸出目錄存在
    output_dir = os.path.dirname(output_path)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # 儲存為 GeoJSON
    gdf_3826.to_file(output_path, driver="GeoJSON")
    print(f"  ✓ 成功匯出: {output_path}")
    print(f"  最終資料筆數: {len(gdf_3826)}")
    
    return gdf_3826

## 處理凱米颱風資料（2024-07-25 14:00）

In [17]:
# 凱米颱風：2024-07-25 14:00
gaemi_gdf = process_rainfall_data(
    csv_path="../data/raw/rain_20240725.csv",
    target_time="2024-07-25 14:00:00",
    output_path="../data/processed/gaemi_rainfall.geojson"
)


處理檔案: ../data/raw/rain_20240725.csv
目標時間: 2024-07-25 14:00:00

[Step 1] 讀取 CSV 檔案...
  原始資料筆數: 162510

[Step 2] 篩選縣市: ['花蓮縣', '宜蘭縣']...
  篩選後資料筆數: 21167

[Step 3] 篩選時間: 2024-07-25 14:00:00...
  時間篩選後資料筆數: 147

[Step 4] 清洗數值（過濾 Past1hr 為 -998 或 <= 0）...
  清洗前資料筆數: 147
  清洗後資料筆數: 117
  移除異常/無雨站點數: 30

[Step 5] 建立 GeoDataFrame 並進行座標轉換...
  原始 CRS: EPSG:4326 (WGS84)
  轉換後 CRS: EPSG:3826 (TWD97 二度分帶)
  已新增 easting 與 northing 欄位

[資料統計]
  Past1hr 平均值: 2.51 mm
  Past1hr 最大值: 9.50 mm
  Past1hr 最小值: 0.50 mm

[Step 6] 匯出至 ../data/processed/gaemi_rainfall.geojson...
  ✓ 成功匯出: ../data/processed/gaemi_rainfall.geojson
  最終資料筆數: 117


In [18]:
# 檢視凱米颱風處理結果
if gaemi_gdf is not None:
    display(gaemi_gdf[['StationName', 'CountyName', 'DateTime', 'Past1hr', 'easting', 'northing']].head(10))

,StationName,CountyName,DateTime,Past1hr,easting,northing
94773,明利國小,花蓮縣,2024-07-25 14:00:00,5.0,292063.325747,2.622568e+06
94774,和仁車站,花蓮縣,2024-07-25 14:00:00,1.5,321993.614354,2.681361e+06
94775,崇德國小,花蓮縣,2024-07-25 14:00:00,1.5,316359.089761,2.673070e+06
94807,大豐,花蓮縣,2024-07-25 14:00:00,4.5,288381.872782,2.611576e+06
94808,高山部落,花蓮縣,2024-07-25 14:00:00,3.0,306034.363139,2.621015e+06
94847,淡大蘭陽校園,宜蘭縣,2024-07-25 14:00:00,3.0,323831.488690,2.746397e+06
94848,大溪漁港,宜蘭縣,2024-07-25 14:00:00,0.5,341189.506987,2.759635e+06
94849,石城,宜蘭縣,2024-07-25 14:00:00,3.0,346034.192412,2.763948e+06
94872,北關,宜蘭縣,2024-07-25 14:00:00,1.0,338232.329313,2.755785e+06
94873,壯圍,宜蘭縣,2024-07-25 14:00:00,0.5,332569.329596,2.736774e+06


## 處理梅雨資料（2024-05-28 14:00）

In [19]:
# 梅雨：2024-05-28 14:00
plumrain_gdf = process_rainfall_data(
    csv_path="../data/raw/rain_20240528.csv",
    target_time="2024-05-28 14:00:00",
    output_path="../data/processed/plumrain_rainfall.geojson"
)


處理檔案: ../data/raw/rain_20240528.csv
目標時間: 2024-05-28 14:00:00

[Step 1] 讀取 CSV 檔案...
  原始資料筆數: 160770

[Step 2] 篩選縣市: ['花蓮縣', '宜蘭縣']...
  篩選後資料筆數: 20563

[Step 3] 篩選時間: 2024-05-28 14:00:00...
  時間篩選後資料筆數: 142

[Step 4] 清洗數值（過濾 Past1hr 為 -998 或 <= 0）...
  清洗前資料筆數: 142
  清洗後資料筆數: 26
  移除異常/無雨站點數: 116

[Step 5] 建立 GeoDataFrame 並進行座標轉換...
  原始 CRS: EPSG:4326 (WGS84)
  轉換後 CRS: EPSG:3826 (TWD97 二度分帶)
  已新增 easting 與 northing 欄位

[資料統計]
  Past1hr 平均值: 0.98 mm
  Past1hr 最大值: 4.00 mm
  Past1hr 最小值: 0.50 mm

[Step 6] 匯出至 ../data/processed/plumrain_rainfall.geojson...
  ✓ 成功匯出: ../data/processed/plumrain_rainfall.geojson
  最終資料筆數: 26


In [20]:
# 檢視梅雨處理結果
if plumrain_gdf is not None:
    display(plumrain_gdf[['StationName', 'CountyName', 'DateTime', 'Past1hr', 'easting', 'northing']].head(10))

,StationName,CountyName,DateTime,Past1hr,easting,northing
93098,寒溪,宜蘭縣,2024-05-28 14:00:00,1.0,322579.028499,2.725429e+06
93102,玉蘭,宜蘭縣,2024-05-28 14:00:00,1.0,309413.305126,2.729939e+06
93109,大農,花蓮縣,2024-05-28 14:00:00,0.5,292177.190010,2.612473e+06
93116,高寮,花蓮縣,2024-05-28 14:00:00,0.5,286470.481858,2.587985e+06
93130,舞鶴,花蓮縣,2024-05-28 14:00:00,0.5,288213.449177,2.596170e+06
93147,水源,花蓮縣,2024-05-28 14:00:00,0.5,304656.511500,2.654226e+06
93180,鳳林山,花蓮縣,2024-05-28 14:00:00,1.0,292831.944047,2.625769e+06
93184,水璉,花蓮縣,2024-05-28 14:00:00,0.5,305282.577310,2.632662e+06
93186,東澳,宜蘭縣,2024-05-28 14:00:00,0.5,334425.221765,2.713085e+06
93241,牛鬥,宜蘭縣,2024-05-28 14:00:00,1.0,308073.848741,2.725786e+06


## 資料處理摘要

In [21]:
print("\n" + "="*60)
print("資料處理完成摘要")
print("="*60)

if gaemi_gdf is not None:
    print(f"\n【凱米颱風】")
    print(f"  輸出檔案: ../data/processed/gaemi_rainfall.geojson")
    print(f"  有效站點數: {len(gaemi_gdf)}")
    print(f"  平均時雨量: {gaemi_gdf['Past1hr'].mean():.2f} mm")
    print(f"  縣市分布:")
    for county, count in gaemi_gdf['CountyName'].value_counts().items():
        print(f"    - {county}: {count} 站")

if plumrain_gdf is not None:
    print(f"\n【梅雨】")
    print(f"  輸出檔案: ../data/processed/plumrain_rainfall.geojson")
    print(f"  有效站點數: {len(plumrain_gdf)}")
    print(f"  平均時雨量: {plumrain_gdf['Past1hr'].mean():.2f} mm")
    print(f"  縣市分布:")
    for county, count in plumrain_gdf['CountyName'].value_counts().items():
        print(f"    - {county}: {count} 站")

print("\n" + "="*60)


資料處理完成摘要

【凱米颱風】
  輸出檔案: ../data/processed/gaemi_rainfall.geojson
  有效站點數: 117
  平均時雨量: 2.51 mm
  縣市分布:
    - 花蓮縣: 82 站
    - 宜蘭縣: 35 站

【梅雨】
  輸出檔案: ../data/processed/plumrain_rainfall.geojson
  有效站點數: 26
  平均時雨量: 0.98 mm
  縣市分布:
    - 宜蘭縣: 14 站
    - 花蓮縣: 12 站

